### 导入包并初始化设备

In [ ]:
# Cell 1: 安装依赖、挂载Google Drive、设置路径


import os
import json
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
#from google.colab import drive
from transformers import BertTokenizer

# 挂载Google Drive
#drive.mount('/content/drive')

# 设置数据路径（请根据实际位置修改）
DATA_PATH = '/content/drive/MyDrive/SQuAD/'
TRAIN_FILE = os.path.join(DATA_PATH, 'SQuAD-train-v2.0.json')
DEV_FILE = os.path.join(DATA_PATH, 'SQuAD-dev-v2.0.json')
CHECKPOINT_DIR = '/content/drive/MyDrive/SQuAD_checkpoints/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 定义数据集类与数据加载器

In [ ]:
# Cell 2: SQuAD Dataset with BertTokenizer

class SQuADDataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length=384, is_train=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        self.examples = self._extract_examples(raw_data)
        self.input_ids = []
        self.attention_mask = []
        self.start_positions = []
        self.end_positions = []
        self._tokenize()
        print(f"Loaded {len(self.examples)} valid examples from {json_path}")

    def _extract_examples(self, raw_data):
        examples = []
        for article in raw_data['data']:
            for para in article['paragraphs']:
                context = para['context']
                for qa in para['qas']:
                    if qa.get('is_impossible', False):
                        continue
                    if not qa.get('answers'):
                        continue
                    ans = qa['answers'][0]
                    examples.append({
                        'context': context,
                        'question': qa['question'],
                        'answer_text': ans['text'],
                        'answer_start_char': ans['answer_start']
                    })
        return examples

    def _tokenize(self):
        skipped = 0
        for ex in tqdm(self.examples, desc="Tokenizing"):
            enc = self.tokenizer(
                ex['question'], ex['context'],
                max_length=self.max_length,
                truncation=True,
                padding='max_length',
                return_tensors='pt',
                return_offsets_mapping=True
            )
            input_ids = enc['input_ids'].squeeze(0)
            att_mask = enc['attention_mask'].squeeze(0)
            offsets = enc['offset_mapping'].squeeze(0).tolist()

            if self.is_train:
                start_char = ex['answer_start_char']
                end_char = start_char + len(ex['answer_text'])
                start_token, end_token = None, None
                for i, (tok_start, tok_end) in enumerate(offsets):
                    if tok_start == 0 and tok_end == 0:
                        continue
                    if tok_start <= start_char < tok_end:
                        start_token = i
                    if tok_start < end_char <= tok_end:
                        end_token = i
                if start_token is None or end_token is None:
                    skipped += 1
                    continue
                self.start_positions.append(start_token)
                self.end_positions.append(end_token)

            self.input_ids.append(input_ids)
            self.attention_mask.append(att_mask)

        print(f"Skipped {skipped} examples (answer span not found)")
        self.input_ids = torch.stack(self.input_ids)
        self.attention_mask = torch.stack(self.attention_mask)
        if self.is_train:
            self.start_positions = torch.tensor(self.start_positions, dtype=torch.long)
            self.end_positions = torch.tensor(self.end_positions, dtype=torch.long)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        if self.is_train:
            return {
                'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'start_pos': self.start_positions[idx],
                'end_pos': self.end_positions[idx]
            }
        else:
            return {
                'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx]
            }


def create_dataloaders(train_path, dev_path, batch_size=16, max_length=384):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    train_dataset = SQuADDataset(train_path, tokenizer, max_length, is_train=True)
    dev_dataset = SQuADDataset(dev_path, tokenizer, max_length, is_train=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, dev_loader, tokenizer

### 实现Transformer基类

In [ ]:
# Cell 3: Transformer 基础组件（支持任意维度 mask）

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # 支持多种维度的 mask
            if mask.dim() == 2:               # (batch, seq_len) -> (batch,1,1,seq_len)
                mask = mask.unsqueeze(1).unsqueeze(2)
            elif mask.dim() == 3:             # (batch, seq_q, seq_k) -> (batch,1,seq_q,seq_k)
                mask = mask.unsqueeze(1)
            # 现在 mask 形状为 (batch, 1, seq_q, seq_k) 或 (batch, nhead, seq_q, seq_k)
            # 若 nhead 维度不匹配，广播
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(context)


class FeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn))
        ff = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None):
        attn = self.self_attn(x, x, x, self_mask)
        x = self.norm1(x + self.dropout1(attn))
        cross = self.cross_attn(x, encoder_output, encoder_output, cross_mask)
        x = self.norm2(x + self.dropout2(cross))
        ff = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff))
        return x

### 实现QA模型1

In [ ]:
# Cell 4: EncoderOnlyQA

class EncoderOnlyQA(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=6,
                 dim_feedforward=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.start_output = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        self.end_output = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        for layer in self.encoder_layers:
            x = layer(x, attention_mask)
        start_logits = self.start_output(x).squeeze(-1)
        end_logits = self.end_output(x).squeeze(-1)
        return start_logits, end_logits

### 实现QA模型2

In [ ]:
# Cell 5: EncoderDecoderQA

class EncoderDecoderQA(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_encoder_layers=4,
                 num_decoder_layers=4, dim_feedforward=1024, max_len=512,
                 dropout=0.1, max_answer_len=50,
                 cls_token_id=101, sep_token_id=102):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.max_answer_len = max_answer_len
        self.cls_token_id = cls_token_id
        self.sep_token_id = sep_token_id

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        self.pos_decoder = PositionalEncoding(d_model, max_len, dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_decoder_layers)
        ])

        self.output_layer = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, vocab_size)
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _get_causal_mask(self, size, device):
        # 下三角为 True（可见），上三角为 False（将被 mask）
        mask = torch.tril(torch.ones(size, size, device=device)).bool()
        return mask

    def encode(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        for layer in self.encoder_layers:
            x = layer(x, attention_mask)
        return x

    def decode(self, decoder_input, encoder_output, cross_mask=None):
        seq_len = decoder_input.size(1)
        device = decoder_input.device
        causal_mask = self._get_causal_mask(seq_len, device)  # (seq_len, seq_len)
        x = self.embedding(decoder_input) * math.sqrt(self.d_model)
        x = self.pos_decoder(x)
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, self_mask=causal_mask, cross_mask=cross_mask)
        return x

    def forward(self, input_ids, decoder_input_ids=None, attention_mask=None):
        enc_out = self.encode(input_ids, attention_mask)
        if decoder_input_ids is not None:
            dec_out = self.decode(decoder_input_ids, enc_out, cross_mask=attention_mask)
            logits = self.output_layer(dec_out)
            return logits
        else:
            return self.generate(enc_out, attention_mask)

    def generate(self, encoder_output, attention_mask=None):
        batch_size = encoder_output.size(0)
        device = encoder_output.device
        decoder_input = torch.full((batch_size, 1), self.cls_token_id, dtype=torch.long, device=device)
        for _ in range(self.max_answer_len):
            dec_out = self.decode(decoder_input, encoder_output, cross_mask=attention_mask)
            next_logits = self.output_layer(dec_out[:, -1, :])
            next_token = next_logits.argmax(dim=-1, keepdim=True)
            decoder_input = torch.cat([decoder_input, next_token], dim=1)
            if (next_token == self.sep_token_id).all():
                break
        return decoder_input

### 实现训练器Trainer

In [ ]:
# Cell 6: 训练器与实验

class QATrainer:
    def __init__(self, model, device=None):
        self.model = model
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.train_losses = []
        self.dev_losses = []
        self.dev_span_accs = []
        self.best_span_acc = 0
        self.best_state = None
        self.is_enc_dec = isinstance(model, EncoderDecoderQA)
        print(f"Trainer on {self.device}, model: {'EncDec' if self.is_enc_dec else 'EncOnly'}")

    def train(self, train_loader, dev_loader, epochs=10, lr=3e-4,
              opt_name='adamw', weight_decay=0.01, grad_clip=1.0, verbose=True):
        opt = self._get_optimizer(opt_name, lr, weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for epoch in range(epochs):
            train_loss = self._train_epoch(train_loader, opt, grad_clip, epoch, epochs, verbose)
            dev_metrics = self.evaluate(dev_loader)
            self.train_losses.append(train_loss)
            self.dev_losses.append(dev_metrics['loss'])
            self.dev_span_accs.append(dev_metrics['span_acc'])
            if dev_metrics['span_acc'] > self.best_span_acc:
                self.best_span_acc = dev_metrics['span_acc']
                self._save_best()
            scheduler.step()
            if verbose:
                print(f"\nEpoch {epoch+1}/{epochs}: Train Loss={train_loss:.4f}, "
                      f"Dev Loss={dev_metrics['loss']:.4f}, Span Acc={dev_metrics['span_acc']:.2f}%, LR={opt.param_groups[0]['lr']:.6f}")
        self._restore_best()
        print(f"\n✅ Best span accuracy: {self.best_span_acc:.2f}%")
        return self

    def _train_epoch(self, loader, opt, grad_clip, epoch, total, verbose):
        self.model.train()
        total_loss = 0
        it = tqdm(loader, desc=f'Epoch {epoch+1}/{total}') if verbose else loader
        for batch in it:
            ids = batch['input_ids'].to(self.device)
            mask = batch['attention_mask'].to(self.device)
            opt.zero_grad()
            if self.is_enc_dec:
                start = batch['start_pos'].to(self.device)
                end = batch['end_pos'].to(self.device)
                # 提取答案 token
                targets = []
                for i in range(ids.size(0)):
                    ans = ids[i, start[i]:end[i]+1]
                    targets.append(ans)
                if any(len(t)==0 for t in targets):
                    continue
                max_len = max(len(t) for t in targets)
                padded = []
                for t in targets:
                    if len(t) < max_len:
                        t = torch.cat([t, torch.full((max_len-len(t),), 0, device=self.device)])
                    padded.append(t)
                targets = torch.stack(padded)           # (B, L)
                dec_in = targets[:, :-1]                # teacher forcing input
                target = targets[:, 1:]                 # prediction target
                if dec_in.size(1) == 0:
                    continue
                logits = self.model(ids, dec_in, mask)  # (B, L-1, vocab)
                loss = nn.CrossEntropyLoss(ignore_index=0)(logits.reshape(-1, self.model.vocab_size), target.reshape(-1))
            else:
                start = batch['start_pos'].to(self.device)
                end = batch['end_pos'].to(self.device)
                s_logit, e_logit = self.model(ids, mask)
                loss = nn.CrossEntropyLoss()(s_logit, start) + nn.CrossEntropyLoss()(e_logit, end)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
            opt.step()
            total_loss += loss.item()
            if verbose:
                it.set_postfix(loss=loss.item())
        return total_loss / max(len(loader), 1)

    def evaluate(self, loader):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(loader, desc="Evaluating", leave=False):
                ids = batch['input_ids'].to(self.device)
                mask = batch['attention_mask'].to(self.device)
                if self.is_enc_dec:
                    start = batch['start_pos'].to(self.device)
                    end = batch['end_pos'].to(self.device)
                    targets = []
                    for i in range(ids.size(0)):
                        ans = ids[i, start[i]:end[i]+1]
                        targets.append(ans)
                    if any(len(t)==0 for t in targets):
                        continue
                    max_len = max(len(t) for t in targets)
                    padded = []
                    for t in targets:
                        if len(t) < max_len:
                            t = torch.cat([t, torch.full((max_len-len(t),), 0, device=self.device)])
                        padded.append(t)
                    targets = torch.stack(padded)
                    dec_in = targets[:, :-1]
                    target = targets[:, 1:]
                    if dec_in.size(1) == 0:
                        continue
                    logits = self.model(ids, dec_in, mask)
                    loss = nn.CrossEntropyLoss(ignore_index=0)(logits.reshape(-1, self.model.vocab_size), target.reshape(-1))
                    total_loss += loss.item()
                    # 生成并评估（去掉开头的 [CLS]）
                    gen = self.model.generate(self.model.encode(ids, mask), mask)
                    for i in range(ids.size(0)):
                        gen_seq = gen[i, 1:]          # 去掉 [CLS]
                        ref_seq = targets[i]
                        min_len = min(len(gen_seq), len(ref_seq))
                        if min_len > 0 and torch.equal(gen_seq[:min_len], ref_seq[:min_len]):
                            correct += 1
                    total += ids.size(0)
                else:
                    start = batch['start_pos'].to(self.device)
                    end = batch['end_pos'].to(self.device)
                    s_logit, e_logit = self.model(ids, mask)
                    loss = nn.CrossEntropyLoss()(s_logit, start) + nn.CrossEntropyLoss()(e_logit, end)
                    total_loss += loss.item()
                    pred_s = s_logit.argmax(dim=-1)
                    pred_e = e_logit.argmax(dim=-1)
                    correct += ((pred_s == start) & (pred_e == end)).sum().item()
                    total += start.size(0)
        return {'loss': total_loss / max(len(loader), 1), 'span_acc': 100.0 * correct / max(total, 1)}

    def _get_optimizer(self, name, lr, wd):
        name = name.lower()
        if name == 'adam':
            return optim.Adam(self.model.parameters(), lr=lr, weight_decay=wd)
        if name == 'adamw':
            return optim.AdamW(self.model.parameters(), lr=lr, weight_decay=wd)
        if name == 'sgd':
            return optim.SGD(self.model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
        raise ValueError(f"Unknown optimizer: {name}")

    def _save_best(self):
        self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}

    def _restore_best(self):
        if self.best_state:
            self.model.load_state_dict(self.best_state)
            self.model.to(self.device)

    def save_model(self, path):
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'train_losses': self.train_losses,
            'dev_losses': self.dev_losses,
            'dev_span_accs': self.dev_span_accs,
            'best_span_acc': self.best_span_acc
        }, path)
        print(f"Saved to {path}")

    def plot_history(self):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(self.train_losses, label='Train Loss')
        axes[0].plot(self.dev_losses, label='Dev Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].grid(True)
        axes[1].plot(self.dev_span_accs, label='Span Accuracy')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy (%)')
        axes[1].legend()
        axes[1].grid(True)
        plt.tight_layout()
        plt.show()




### 实验

In [ ]:
# 主实验
if __name__ == "__main__":
    print("="*70)
    print("SQuAD QA with Custom Transformers")
    print("="*70)

    BATCH_SIZE = 16
    MAX_LEN = 384
    EPOCHS = 3   # 可改为 5 或 10

    train_loader, dev_loader, tokenizer = create_dataloaders(
        TRAIN_FILE, DEV_FILE, batch_size=BATCH_SIZE, max_length=MAX_LEN
    )
    vocab_size = tokenizer.vocab_size
    print(f"Vocabulary size: {vocab_size}")

    # Encoder-Only
    print("\n" + "="*70)
    print("Experiment 1: Encoder-Only")
    model1 = EncoderOnlyQA(vocab_size, d_model=128, nhead=8, num_layers=4,
                           dim_feedforward=512, max_len=MAX_LEN, dropout=0.1)
    print(f"Params: {sum(p.numel() for p in model1.parameters() if p.requires_grad):,}")
    trainer1 = QATrainer(model1, device)
    trainer1.train(train_loader, dev_loader, epochs=EPOCHS, lr=3e-4, opt_name='adamw')
    trainer1.save_model(os.path.join(CHECKPOINT_DIR, "EncoderOnlyQA.pt"))
    trainer1.plot_history()

    # Encoder-Decoder
    print("\n" + "="*70)
    print("Experiment 2: Encoder-Decoder")
    model2 = EncoderDecoderQA(vocab_size, d_model=128, nhead=8,
                              num_encoder_layers=3, num_decoder_layers=3,
                              dim_feedforward=512, max_len=MAX_LEN, dropout=0.1,
                              max_answer_len=50,
                              cls_token_id=tokenizer.cls_token_id,
                              sep_token_id=tokenizer.sep_token_id)
    print(f"Params: {sum(p.numel() for p in model2.parameters() if p.requires_grad):,}")
    trainer2 = QATrainer(model2, device)
    trainer2.train(train_loader, dev_loader, epochs=EPOCHS, lr=3e-4, opt_name='adamw')
    trainer2.save_model(os.path.join(CHECKPOINT_DIR, "EncoderDecoderQA.pt"))
    trainer2.plot_history()

    print("\n" + "="*70)
    print("Final Results")
    print(f"Encoder-Only    Best Span Acc: {trainer1.best_span_acc:.2f}%")
    print(f"Encoder-Decoder Best Span Acc: {trainer2.best_span_acc:.2f}%")
    print("="*70)